In [1]:
################################################################################
## CPI MULTI-YEAR CONSOLIDATED EXTRACTION PIPELINE (2013 - 2026)
## CPI CODE-SCHEMA EXTRACTION PIPELINE USING esankhyiki
##
## Patched V3:
## - Uses CPI coded parameter schema throughout
## - Parent/group/subgroup extraction uses coded CPI params
## - Saves yearly interim files in INTERIM_DIR
## - Consolidates master file into PROCESSED_DIR (proc-data)
## - Automatically cleans up/deletes interim files after successful consolidation
##
## Pre-Requisite:
##     pip install mospi-esankhyiki pandas openpyxl
################################################################################
import os
import time
from datetime import datetime
import pandas as pd
import urllib3
import warnings

# Suppress unverified SSL request warnings from MoSPI servers
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings("ignore", category=urllib3.exceptions.InsecureRequestWarning)

import esankhyiki
from esankhyiki.exceptions import (
    InvalidDatasetError,
    InvalidFilterError,
    APIError,
    NoDataError,
)

# --------------------------------------------
# PRINT START TIME
# --------------------------------------------
code_start_time = datetime.now()
timer_start = time.time()
print("Code run start time:", code_start_time.strftime("%d/%m/%Y %H:%M:%S"))

# --------------------------------------------
# ========= CONFIGURE PATHS & STUBS =========
# --------------------------------------------
proj_stub = r"/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/data/"
stub = r"cpi-mospi"
byear_stub = r"2012"  # Standard CPI Base Year

BASE_DIR = f"{proj_stub}"
RAW_DIR = os.path.join(BASE_DIR, f"raw-data/{stub}")
INTERIM_DIR = os.path.join(BASE_DIR, "interim-data")
PROCESSED_DIR = os.path.join(BASE_DIR, "proc-data")

os.makedirs(INTERIM_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(RAW_DIR, exist_ok=True)

# CPI RAW & CONSOLIDATED FILE LOCATIONS
cpi_metadata_file = os.path.join(RAW_DIR, f"{stub}-b{byear_stub}-metadata.xlsx")
cpi_master_data_file = os.path.join(PROCESSED_DIR, f"{stub}-b{byear_stub}-data-consolidated.xlsx")

# --------------------------------------------
# STEP 1, 2 & 3: DISCOVERY & METADATA
# --------------------------------------------
try:
    datasets_df = esankhyiki.list_datasets(format="df")
    indicators_df = esankhyiki.get_indicators("CPI", format="df")

    # Fetch CPI Metadata structure
    metadata_df = esankhyiki.get_metadata(
        "CPI",
        base_year=f"{byear_stub}",
        series="Current",
        format="df"
    )
    with pd.ExcelWriter(cpi_metadata_file, engine="openpyxl") as writer:
        metadata_df.to_excel(writer, sheet_name="CPI-METADATA", index=False)
    print(f"--> Saved CPI Metadata to: {cpi_metadata_file}")
except Exception as e:
    print(f"Notice during metadata extraction: {e}")


# ------------------------------------------------------------------------------
# THE SUPER-LOOP: Iterate from 2013 to 2026
# ------------------------------------------------------------------------------

# Master container for absolute consolidation across all years
global_compiled_records = []

# Tracker for created interim files to enable cleanup after consolidation
interim_file_paths = []

# CPI Base 2012 series begins effectively from 2013
start_year = 2013
end_year = 2026

for current_year in range(start_year, end_year + 1):
    print(f"\n====================================================")
    print(f"STARTING CPI EXTRACTION FOR YEAR: {current_year}")
    print(f"====================================================")
    
    # Yearly container for isolating single-year data files
    year_compiled_records = []
    
    # Loop through months 1 to 12
    for month in range(1, 13):
        month_str = str(month).zfill(2)
        page = 1
        max_pages_per_month = 15  
        
        print(f"--- Fetching CPI Data for {current_year} | Month: {month_str} ---")
        
        while page <= max_pages_per_month:
            try:
                print(f"Requesting Year {current_year} | Month {month_str} | Page {page}...")
                
                # CPI data request filters
                cpi_filters = {
                    "series": "Current",        # Required parameter for CPI
                    "base_year": f"{byear_stub}",
                    "sector_code": 3,           # 3 = Combined (1 = Rural, 2 = Urban)
                    "year": current_year,
                    "month_code": month,
                    "limit": 100,
                    "page": page
                }
                
                monthly_page_df = esankhyiki.get_data("CPI", cpi_filters, format="df")
                
                # Handle empty page blocks
                if monthly_page_df is None or monthly_page_df.empty:
                    print(f"No more data found for month {month_str} at page {page}.")
                    break
                
                # Add tracking columns for provenance
                monthly_page_df['extracted_year'] = current_year
                monthly_page_df['extracted_month'] = month
                
                # Append to both local year and global master lists
                year_compiled_records.append(monthly_page_df)
                global_compiled_records.append(monthly_page_df)
                
                page += 1
                time.sleep(1)  # Throttle to prevent server overload
                
            except NoDataError:
                print(f"Reached final page index for Month {month_str} (NoDataError triggered).")
                break
            except APIError as e:
                print(f"Server connection drop on Year {current_year}, Month {month_str}, Page {page}: {e}")
                break
            except Exception as e:
                print(f"Unexpected error encountered: {e}")
                break

    # Save individual year file into INTERIM_DIR right after its months complete
    if year_compiled_records:
        year_df = pd.concat(year_compiled_records, ignore_index=True)
        yearly_output_file = os.path.join(INTERIM_DIR, f"{stub}-data-{current_year}-b{byear_stub}.xlsx")
        
        with pd.ExcelWriter(yearly_output_file, engine="openpyxl") as writer:
            year_df.to_excel(writer, sheet_name=f"CPI-{current_year}", index=False)
            
        interim_file_paths.append(yearly_output_file)
        print(f"--> Saved interim snapshot file for {current_year}: {yearly_output_file}")
    else:
        print(f"--> No CPI records recovered or available for the year {current_year}.")

# ------------------------------------------------------------------------------
# STEP 5: FINAL CONSOLIDATION, MASTER OUTPUT & INTERIM CLEANUP
# ------------------------------------------------------------------------------
print(f"\n====================================================")
print("RUN COMPLETE. BUILDING MASTER CONSOLIDATED CPI FILE...")
print(f"====================================================")

if global_compiled_records:
    final_master_df = pd.concat(global_compiled_records, ignore_index=True)
    print(f"\nSuccess! Total row count across entire CPI dataset ({start_year}-{end_year}): {len(final_master_df)}")
    
    # Save the consolidated master file in PROCESSED_DIR (proc-data)
    with pd.ExcelWriter(cpi_master_data_file, engine="openpyxl") as writer:
        final_master_df.to_excel(writer, sheet_name="CPI-CONSOLIDATED-MASTER", index=False)
        
    print(f"Grand master consolidated file securely saved to: {cpi_master_data_file}")
    
    # --------------------------------------------
    # CLEANUP: Remove interim yearly files
    # --------------------------------------------
    print(f"\n----------------------------------------------------")
    print("CLEANING UP INTERIM FILES FROM INTERIM-DATA FOLDER...")
    print(f"----------------------------------------------------")
    for interim_file in interim_file_paths:
        if os.path.exists(interim_file):
            try:
                os.remove(interim_file)
                print(f"--> Successfully deleted interim file: {interim_file}")
            except Exception as cleanup_err:
                print(f"--> Warning: Could not delete interim file {interim_file}: {cleanup_err}")
    print("Interim file cleanup completed.")

else:
    print("\nExtraction critical failure: No CPI data blocks were generated across any year parameters.")

# --------------------------------------------
# PRINT END TIME
# --------------------------------------------
print('finito')
code_end_time = datetime.now()
timer_end = time.time()
print("Code run end time:", code_end_time.strftime("%d/%m/%Y %H:%M:%S"))
print("Total multi-year execution elapsed time (seconds):", round(timer_end - timer_start, 2))

Code run start time: 29/07/2026 13:25:00
--> Saved CPI Metadata to: /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/data/raw-data/cpi-mospi/cpi-mospi-b2012-metadata.xlsx

STARTING CPI EXTRACTION FOR YEAR: 2013
--- Fetching CPI Data for 2013 | Month: 01 ---
Requesting Year 2013 | Month 01 | Page 1...
Requesting Year 2013 | Month 01 | Page 2...
Requesting Year 2013 | Month 01 | Page 3...
Requesting Year 2013 | Month 01 | Page 4...
Requesting Year 2013 | Month 01 | Page 5...
Requesting Year 2013 | Month 01 | Page 6...
Requesting Year 2013 | Month 01 | Page 7...
Requesting Year 2013 | Month 01 | Page 8...
Requesting Year 2013 | Month 01 | Page 9...
Reached final page index for Month 01 (NoDataError triggered).
--- Fetching CPI Data for 2013 | Month: 02 ---
Requesting Year 2013 | Month 02 | Page 1...
Requesting Year 2013 | Month 02 | Page 2...
Requesting Year 2013 | Month 02 | Page 3...
Requesting Year 2013 | Month 02 | Page 4...
Requesting Yea